###### **ÉTAPE 1 - IMPORTS**

In [110]:
import pandas as pd
import io
from google.cloud import bigquery, storage
from google.api_core.exceptions import Conflict  # pour gérer le dataset déjà existant
import warnings
warnings.filterwarnings("ignore")

###### **ÉTAPE 2 - CONFIGURATION**



In [111]:
# Paramètres du projet
project_id = "randstad-data-platform"
dataset_cible = "fact_transactions"
bucket_name = "db_randstad"
folder = "external"

In [112]:
# Noms des fichiers dans le bucket
fichier_s5 = "S5_sinistres_as400_fixedwidth.txt"
fichier_s6 = "S6_marche_bloomberg_ticks.csv"
fichier_s7 = "S7_agences_commerciales.xlsx"

In [113]:
# Connexion aux services GCP, dans bigquery notebooks, l'authentification est automatique
bq_client = bigquery.Client(project=project_id)
gcs_client = storage.Client(project=project_id)

In [114]:
print("Connexion GCP établie")
print(f" Projet : {project_id}")
print(f" Dataset : {dataset_cible}")
print(f" Bucket : {bucket_name}/{folder}")

Connexion GCP établie
 Projet : randstad-data-platform
 Dataset : fact_transactions
 Bucket : db_randstad/external


**ÉTAPE 3 - CRÉATION INTELLIGENTE DU DATASET (anti-conflit)**
###### C'est ici qu'on évite les conflits lors de la fusion des ingestions :

*   Si 'fact_transactions' n'existe pas encore => on le crée
*   Si 'fact_transactions' existe déjà (créé par un coéquipier) => on continue sans rien écraser

In [115]:
def creer_dataset_si_absent(dataset_id) :
  """
  Crée un dataset BigQuery uniquement s'il n'existe pas déjà.
  Évite les conflits quand plusieurs membres de l'équipe exécutent leur script.

  Arguments : dataset_id : nom du dataset à créer (ex: 'fact_transactions')
  """
  dataset_ref = f"{project_id}.{dataset_id}"
  dataset = bigquery.Dataset(dataset_ref)
  dataset.location = "US" #même région que le bucket

  try :
    bq_client.create_dataset(dataset)
    print(f" ✅ Dataset '{dataset_id}' crée avec succès")
  except Conflict :
    # le dataset existe déjà, c'est normal et on ne fait rien
    print( f" ℹ️ Dataset '{dataset_id}' existe déjà, on continue sans l'écraser")

In [116]:
print("\n🗂️  Vérification du dataset...")


🗂️  Vérification du dataset...


In [117]:
creer_dataset_si_absent(dataset_cible)

 ℹ️ Dataset 'fact_transactions' existe déjà, on continue sans l'écraser


###### **ÉTAPE 4 — FONCTIONS UTILITAIRES**

In [118]:
from google.cloud.storage import blob
def lire_depuis_gcs(nom_fichier, mode="bytes"):
  """
  Lit un fichier depuis Google Cloud Storage et le retourne en mémoire.

  Arguments :
  * nom_fichier : nom du fichier dans le bucket (ex: 'S6_marche_bloomberg_ticks.csv')
  * mode : 'bytes' pour fichiers binaires (xlsx), 'text' pour texte (csv, txt)
  """
  chemin_complet = f"{folder}/{nom_fichier}"
  bucket = gcs_client.bucket(bucket_name)
  blob = bucket.blob(chemin_complet)
  contenu = blob.download_as_bytes()

  print(f" 📥 Fichier téléchargé depuis gs://{bucket_name}/{chemin_complet}")

  if mode == "text":
    return io.StringIO(contenu.decode("latin-1"))
  return io.BytesIO(contenu)


In [ ]:
from os import write
from google.cloud.bigquery import table
def charger_dans_bigquery(df, nom_table):
  """
  Envoie un DataFrame vers une table BigQuery.
  Si la table existe déjà, elle est remplacée (WRITE_TRUNCATE).

  Arguments :
  * df : le DataFrame à envoyer
  * nom_table : nom de la table destination (ex: 'raw_sinistres_as400')
  """
  table_id = f"{project_id}.{dataset_cible}.{nom_table}"
  job_config = bigquery.LoadJobConfig(
      write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE, # remplace la table si elle existe déjà
      autodetect = True # laisse BigQuery deviner les types de colonnes automatiquement
  )
  job = bq_client.load_table_from_dataframe(df, table_id, job_config=job_config)
  job.result()
  print(f" ✅ Chargé dans BigQuery => {table_id} ({len(df)} lignes)")


**ÉTAPE 5 — S5 : SINISTRES AS400 (FIXED-WIDTH)**
###### Format fixed-width : chaque colonne occupe un nombre fixe de caractères.
⚠️ Les positions (colspecs) sont à ajuster selon la vraie structure du fichier.
* Pour les vérifier : télécharge le fichier et ouvre-le dans un éditeur de texte,
* puis compte les caractères de chaque colonne.

In [120]:
print("\n📂 S5 — Sinistres AS400...")


📂 S5 — Sinistres AS400...


In [121]:
# Position (début, fin) de chaque colonne dans le fichier
colspecs_s5 = [
    (0, 10),    # ID_SINISTRE
    (10, 20),   # STDATE_SIN
    (20, 28),   # DATE_CLO
    (28, 43),   # MONTANT_INDEMNIS
    (43, 56),   # CODE_POLICE
    (56, 62),   # EXPERT
    (62, None), # LIBELLE_SINISTRE
]

In [122]:
noms_colonnes_s5 = [
    "ID_SINISTRE",
    "STDATE_SIN",
    "DATE_CLO",
    "MONTANT_INDEMNIS",
    "CODE_POLICE",
    "EXPERT",
    "LIBELLE_SINISTRE"
]

In [123]:
contenu_s5 = lire_depuis_gcs(fichier_s5, mode="text")

 📥 Fichier téléchargé depuis gs://db_randstad/external/S5_sinistres_as400_fixedwidth.txt


In [124]:
df_s5 = pd.read_fwf(
    contenu_s5,
    colspecs=colspecs_s5,
    names=noms_colonnes_s5,
    skiprows=1   # ignore la première ligne si c'est un en-tête
)

In [125]:
# Nettoyage
df_s5 = df_s5.dropna(how="all")
for col in df_s5.select_dtypes(include="object").columns:
    df_s5[col] = df_s5[col].str.strip()

In [126]:
print(f"   Lignes : {df_s5.shape[0]} | Colonnes : {df_s5.shape[1]}")
print(f"   Nulls  : {df_s5.isnull().sum().sum()} | Doublons : {df_s5.duplicated().sum()}")

   Lignes : 200000 | Colonnes : 7
   Nulls  : 0 | Doublons : 0


In [127]:
charger_dans_bigquery(df_s5, "fact_transactions_sinistres_as400")

BadRequest: 400 POST https://bigquery.googleapis.com/upload/bigquery/v2/projects/randstad-data-platform/jobs?uploadType=multipart: Invalid project ID ' randstad-data-platform'. Project IDs must contain 6-63 lowercase letters, digits, or dashes. Some project IDs also include domain name separated by a colon. IDs must start with a letter and may not end with a dash.